# Topological MQA (TMQA) 2: `FakeMiami`
This is the second notebook for Topological MQA series. We will run the same process on `FakeMiami`, which is introduced as a part of [`FakeBackendV2`](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/fake-provider-fake-miami)

`FakeMiami` has 120 qubits with 10 x 12 arrangement. Here, FMQA will demonstrate TMQA with its square lattice.

In [ ]:
from qiskit_ibm_runtime.fake_provider.backends.miami import FakeMiami

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import os, random, gc, pickle, pathlib
from collections import defaultdict
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.transpiler import Target
from qiskit.quantum_info import Operator
from qiskit_device_benchmarking.bench_code.mrb import (
    MirrorQATopo,
    QuantumAwesomeness,
    TopoUtil,
)

In [ ]:
SEED = 123
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

shots = 1000
lengths = [2, 6, 14, 24, 30]
num_samples = 100
num_lengths = len(lengths)

In [ ]:
miami_hw = FakeMiami()
leqit = miami_hw.num_qubits # 120 qubits
faqe = leqit + 2

clifford_noise = NoiseModel.from_backend(miami_hw, thermal_relaxation=False)

miami_sim = AerSimulator.from_backend(miami_hw, noise_model=clifford_noise)
miami_sim.set_options(
    method='stabilizer',
    seed_simulator=SEED,
    # max_parallel_threads=0, # Use all available CPU cores
    max_parallel_experiments=0, # Run multiple circuits in parallel
)

In [ ]:
exp = MirrorQATopo(
    range(leqit),
    lengths=lengths,
    sampling_algorithm='topo',
    mode='random',
    backend=miami_sim,
    two_qubit_gate_density=0.5,
    num_samples=num_samples,
    initial_entangling_angle=np.pi / 2,
    seed=SEED,
)

exp.set_run_options(shots=shots)
exp.set_transpile_options(optimization_level=0)

### Import Helper functions

Below cells are for helper functions to:
1. Save the ongoing experiment as `.pkl` to the disk, so if the kernel crashes or the laptop goes to Sleep Mode, a researcher doesn't need to execute the entire experiment from the beginning again.
2. Analysis & Terminal printing: Definition of a *bot*, calculation of the `P(pairs)` and `P(topo)`.

In [ ]:
# Part 1: Prepare .pkl
ckpt_dir = pathlib.Path("topo_ckpt")
ckpt_dir.mkdir(exist_ok=True)

In [ ]:
# Part 2: Experiment side
# To deduct the mode for TopoSampler
def get_topo_mode(topo_full, fake_qubits):
    """f2f if the fake qubits paired together (or stayed unpaired); 
    f2g otherwise."""
    fake_pairs = [p for p in topo_full if any(q in fake_qubits for q in p)]
    if not fake_pairs:
        return "f2f"
    for p in fake_pairs:
        if all(q in fake_qubits for q in p):
            return "f2f"
    return "f2g"

In [ ]:
# Run the experiment
def run_topo(backend, legit_qubits):
    """Build the MirrorQATopo experiment, run it, 
    and block until results land."""
    exp = MirrorQATopo(
        range(legit_qubits),
        lengths=lengths,
        sampling_algorithm="topo",
        mode="random",
        backend=backend,
        two_qubit_gate_density=0.5,
        num_samples=num_samples,
        initial_entangling_angle=np.pi / 2,
        seed=SEED,
    )
    exp.set_run_options(shots=shots)
    rb_data = exp.run()
    
    rb_data.block_for_results()
    return exp, rb_data

In [ ]:
# Calculates the MI from the result
def extract_mi(exp, rb_data, legit_qubits):
    legit_cmap = exp.backend.coupling_map.reduce(range(legit_qubits))
    qa = QuantumAwesomeness(legit_cmap)
    return qa.mutual_info(rb_data.data())

This is a bot.

In [ ]:
def evaluate_bot(mi, exp, legit_qubits, num_qubits, verbose=False):
    half = legit_qubits // 2
    fake_qubits = set(range(legit_qubits, num_qubits))

    bot_mode_ok = defaultdict(int)
    bot_pairs_ok = defaultdict(int)
    totals = defaultdict(int)

    for i, info in enumerate(mi):
        length = lengths[i % num_lengths]
        # sample = i // num_lengths

        topo_full = exp._topo_outcomes[i]
        topo_mode = get_topo_mode(topo_full, fake_qubits)

        G = nx.Graph()
        for (u, v), w in info.items():
            G.add_edge(u, v, weight=w)
        raw_guess = nx.max_weight_matching(G, maxcardinality=False, weight="weight")
        bot_pairs = sorted(tuple(sorted(p)) for p in raw_guess)
        bot_mode = "f2f" if len(bot_pairs) == half else "f2g"

        pairs_truth = sorted(tuple(sorted(p)) for p in exp._pairs[i])

        bot_mode_ok[length] += int(bot_mode == topo_mode)
        bot_pairs_ok[length] += int(set(bot_pairs) == set(pairs_truth))
        totals[length] += 1

    L = sorted(lengths)
    return {
        "lengths": L,
        "p_mode_topo": [bot_mode_ok[l] / totals[l] for l in L],
        "p_bot_pairs": [bot_pairs_ok[l] / totals[l] for l in L],
    }


## Run the experiment!

In [ ]:
rb_data = exp.run()
rb_data.block_for_results()

In [ ]:
mi = extract_mi(exp, rb_data, leqit)
results = {}
results['miami'] = evaluate_bot(mi, exp, leqit, faqe, verbose=False)

In [ ]:
# A plot for P(pairs)
plt.figure(figsize=(7, 4.5))
y = [
    1 - p for p in results["miami"]["p_bot_pairs"]
]
plt.plot(
    results["miami"]["lengths"],
    y,
    marker="o",
    linestyle="-",
    label="FakeMiami",
)
plt.xlabel("Circuit Depth (length)")
plt.ylabel(r"1-$P(\mathrm{pairs})$")
plt.grid(True, alpha=0.3)
plt.legend(loc="center right", frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# A plot for P(topo)
plt.figure(figsize=(7, 4.5))
y = [
    1 - p for p in results["miami"]["p_mode_topo"]
]
plt.plot(
    results["miami"]["lengths"],
    y,
    marker="o",
    linestyle="-",
    label="FakeMiami",
)
plt.xlabel("Circuit Depth (length)")
plt.ylabel(r"1-$P(\mathrm{topo})$")
plt.grid(True, alpha=0.3)
plt.legend(loc="center right", frameon=True)
plt.tight_layout()
plt.show()